# Stress-Strain Profile Analysis

This notebook demonstrates and validates the `StressStrainProfile` class for analyzing strain and stress distributions across reinforced concrete cross-sections.

**Key Concepts:**
- Plane section assumption: ε(z) = ε_bottom + κ × z
- Strain distribution based on curvature κ and reference strain
- Stress profiles in concrete and reinforcement
- Force resultants (N, M) calculation
- Visualization with coupled strain-stress plots

In [ ]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt

# Phase 1: Material models
from bmcs_cross_section.matmod import EC2Concrete, SteelReinforcement, create_steel

# Phase 2: Cross-section design
from bmcs_cross_section.cs_design import (
    RectangularShape,
    BarReinforcement,
    ReinforcementLayout,
    CrossSection,
    StressStrainProfile
)

# Phase 2: Component catalogs
from bmcs_cross_section.cs_components import SteelRebarComponent

print("✓ All imports successful")

## 1. Create Cross-Section

Define a rectangular cross-section with:
- Concrete: C30/37
- Reinforcement: 4×D20 steel rebars (B500B)
- Dimensions: 300mm × 500mm

In [ ]:
# Define shape
shape = RectangularShape(b=300, h=500)
print(f"Shape: {shape.b}mm × {shape.h}mm")
print(f"Area: {shape.area:.0f} mm²")
print(f"Second moment of area: {shape.I_y/1e6:.2f} × 10⁶ mm⁴")

In [ ]:
# Define concrete material
concrete = EC2Concrete(f_ck=30)  # C30/37
print(f"Concrete: C{concrete.f_ck}/{concrete.f_cm}")
print(f"Design strength f_cd: {concrete.f_cd:.2f} MPa")
print(f"Elastic modulus E_cm: {concrete.E_cm/1000:.1f} GPa")

In [ ]:
# Define reinforcement using catalog components
steel_rebar = SteelRebarComponent(nominal_diameter=20, grade='B500B')
print(f"Rebar: D{steel_rebar.nominal_diameter} {steel_rebar.grade}")
print(f"Area per bar: {steel_rebar.area:.1f} mm²")
print(f"Yield strength: {steel_rebar.matmod.f_sy:.0f} MPa")

# Create reinforcement layout
c_nom = 25  # mm cover
reinforcement = ReinforcementLayout()

# Bottom layer (tension zone)
z_bottom = shape.h - c_nom - steel_rebar.nominal_diameter / 2
layer_bottom = BarReinforcement(z=z_bottom, component=steel_rebar, count=4)
reinforcement.layers.append(layer_bottom)

# Top layer (compression zone)
z_top = c_nom + steel_rebar.nominal_diameter / 2
layer_top = BarReinforcement(z=z_top, component=steel_rebar, count=2)
reinforcement.layers.append(layer_top)

print(f"\nBottom layer: 4 bars at z={z_bottom:.1f}mm, A_s={layer_bottom.A_s:.0f} mm²")
print(f"Top layer: 2 bars at z={z_top:.1f}mm, A_s={layer_top.A_s:.0f} mm²")
print(f"Total reinforcement: {reinforcement.total_area:.0f} mm²")

In [ ]:
# Assemble cross-section
cs = CrossSection(
    shape=shape,
    concrete=concrete,
    reinforcement=reinforcement
)

print(f"Cross-section created:")
print(f"  Total height: {cs.h_total:.0f} mm")
print(f"  Concrete area: {cs.shape.A:.0f} mm²")
print(f"  Steel area: {cs.reinforcement.total_area:.0f} mm²")
print(f"  Reinforcement ratio: {cs.reinforcement.total_area / cs.shape.A * 100:.2f}%")

## 2. Define Loading State

Specify the strain state using:
- **Curvature κ**: Rate of change of strain with height [1/mm]
- **Bottom strain ε_bottom**: Reference strain at the bottom of the section [-]

Using plane section assumption: **ε(z) = ε_bottom + κ × z**

In [ ]:
# Define loading state
kappa = -0.00001  # 1/mm (negative = compression at top)
eps_bottom = 0.002  # Bottom strain (tension)

print(f"Loading state:")
print(f"  Curvature κ: {kappa*1000:.3f} × 10⁻³ mm⁻¹")
print(f"  Bottom strain ε_bottom: {eps_bottom:.6f}")
print(f"  Top strain ε_top: {eps_bottom + kappa * cs.h_total:.6f}")

## 3. Create Stress-Strain Profile

Instantiate the `StressStrainProfile` class to compute distributions and forces.

In [ ]:
# Create stress-strain profile
profile = StressStrainProfile(cs, kappa, eps_bottom)

print(f"Stress-Strain Profile created")
print(f"  Cross-section: {cs.shape.b}×{cs.shape.h} mm")
print(f"  Curvature: {profile.kappa*1000:.3f} × 10⁻³ mm⁻¹")
print(f"  Reference strain: {profile.eps_bottom:.6f}")

## 4. Compute Strain Distribution

Calculate strain at various heights using the plane section assumption.

In [ ]:
# Get strain profile
z_coords, strains = profile.get_strain_profile(n_points=100)

print(f"Strain profile computed:")
print(f"  Number of points: {len(z_coords)}")
print(f"  Height range: {z_coords.min():.0f} to {z_coords.max():.0f} mm")
print(f"  Strain range: {strains.min():.6f} to {strains.max():.6f}")

# Strain at specific locations
eps_top = profile.get_strain_at_z(0)
eps_mid = profile.get_strain_at_z(cs.h_total / 2)
eps_bottom_calc = profile.get_strain_at_z(cs.h_total)

print(f"\nStrain at specific locations:")
print(f"  Top (z=0): {eps_top:.6f}")
print(f"  Mid (z={cs.h_total/2:.0f}): {eps_mid:.6f}")
print(f"  Bottom (z={cs.h_total:.0f}): {eps_bottom_calc:.6f}")

In [ ]:
# Plot strain distribution
fig, ax = plt.subplots(figsize=(6, 8))

ax.plot(strains * 1000, z_coords, 'b-', linewidth=2.5, label='Strain profile')
ax.axvline(x=0, color='k', linestyle='--', alpha=0.5)

# Mark reinforcement layers
for layer in cs.reinforcement.layers:
    eps = profile.get_strain_at_z(layer.z)
    ax.plot(eps * 1000, layer.z, 'ro', markersize=10)
    ax.text(eps * 1000 + 0.2, layer.z, f'z={layer.z:.0f}mm\nε={eps:.4f}', fontsize=9)

ax.set_xlabel('Strain [‰]', fontsize=12, fontweight='bold')
ax.set_ylabel('Height z [mm]', fontsize=12, fontweight='bold')
ax.set_title('Strain Distribution (Plane Section Assumption)', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, cs.h_total)

plt.tight_layout()
plt.show()

## 5. Compute Stress Distribution

Calculate stresses in concrete and reinforcement using material constitutive laws.

In [ ]:
# Get concrete stress profile
z_stress, sig_concrete = profile.get_concrete_stress_profile(n_points=100)

print(f"Concrete stress profile:")
print(f"  Stress range: {sig_concrete.min():.2f} to {sig_concrete.max():.2f} MPa")
print(f"  Max compression: {abs(sig_concrete.min()):.2f} MPa")
print(f"  Max tension: {sig_concrete.max():.2f} MPa")

In [ ]:
# Get reinforcement strains and stresses
z_reinf, eps_reinf, sig_reinf = profile.get_reinforcement_strains_stresses()

print(f"\nReinforcement stresses:")
for i, (layer, z, eps, sig) in enumerate(zip(cs.reinforcement.layers, z_reinf, eps_reinf, sig_reinf)):
    print(f"  Layer {i+1} (z={z:.0f}mm):")
    print(f"    Strain: {eps:.6f}")
    print(f"    Stress: {sig:.2f} MPa")
    print(f"    Force: {layer.get_force(eps)/1000:.2f} kN")

In [ ]:
# Plot stress distribution
fig, ax = plt.subplots(figsize=(6, 8))

ax.plot(sig_concrete, z_stress, 'b-', linewidth=2.5, label='Concrete stress')
ax.fill_betweenx(z_stress, 0, sig_concrete, alpha=0.2, color='blue')
ax.axvline(x=0, color='k', linestyle='--', alpha=0.5)

# Mark reinforcement layers
for i, (z, eps, sig) in enumerate(zip(z_reinf, eps_reinf, sig_reinf)):
    color = 'red' if sig > 0 else 'darkred'
    ax.plot(sig, z, 'o', markersize=12, color=color, markeredgecolor='black', markeredgewidth=2)
    ax.text(sig + 20, z, f'σ={sig:.1f} MPa', fontsize=9, va='center')

ax.set_xlabel('Stress σ [MPa]', fontsize=12, fontweight='bold')
ax.set_ylabel('Height z [mm]', fontsize=12, fontweight='bold')
ax.set_title('Stress Distribution', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, cs.h_total)

plt.tight_layout()
plt.show()

## 6. Calculate Force Resultants

Compute concrete force, steel force, total axial force, and bending moment.

In [ ]:
# Calculate force resultants
F_c, F_s, N_total, M_total = profile.get_force_resultants()

print(f"Force Resultants:")
print(f"  Concrete force F_c: {F_c/1000:.2f} kN")
print(f"  Steel force F_s: {F_s/1000:.2f} kN")
print(f"  Total axial force N: {N_total/1000:.2f} kN")
print(f"  Bending moment M: {M_total/1e6:.2f} kNm")

# Check equilibrium
equilibrium_error = abs(F_c + F_s - N_total)
print(f"\nEquilibrium check:")
print(f"  F_c + F_s = {(F_c + F_s)/1000:.2f} kN")
print(f"  N_total = {N_total/1000:.2f} kN")
print(f"  Error: {equilibrium_error:.6f} N (should be ≈0)")

## 7. Comprehensive Visualization

Use the built-in plotting method to show coupled strain-stress profiles with force resultants.

In [ ]:
# Create comprehensive visualization
fig, (ax_strain, ax_stress) = plt.subplots(1, 2, figsize=(14, 8), sharey=True)
profile.plot_stress_strain_profile(ax_strain, ax_stress, show_resultants=True)

plt.tight_layout()
plt.show()

## 8. Parametric Study: Varying Curvature

Explore how force resultants change with curvature.

In [ ]:
# Parametric study: vary curvature
kappa_values = np.linspace(-0.00002, -0.000005, 10)
eps_bottom_fixed = 0.002

results = []
for kappa_i in kappa_values:
    profile_i = StressStrainProfile(cs, kappa_i, eps_bottom_fixed)
    F_c, F_s, N, M = profile_i.get_force_resultants()
    results.append({
        'kappa': kappa_i * 1000,  # × 10⁻³
        'F_c': F_c / 1000,  # kN
        'F_s': F_s / 1000,  # kN
        'N': N / 1000,  # kN
        'M': M / 1e6  # kNm
    })

# Convert to arrays
kappa_plot = [r['kappa'] for r in results]
F_c_plot = [r['F_c'] for r in results]
F_s_plot = [r['F_s'] for r in results]
N_plot = [r['N'] for r in results]
M_plot = [r['M'] for r in results]

# Plot results
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))

ax1.plot(kappa_plot, F_c_plot, 'b-o', linewidth=2, markersize=6)
ax1.set_xlabel('Curvature κ [×10⁻³ mm⁻¹]', fontsize=11)
ax1.set_ylabel('Concrete Force F_c [kN]', fontsize=11)
ax1.set_title('Concrete Force vs Curvature', fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(kappa_plot, F_s_plot, 'r-o', linewidth=2, markersize=6)
ax2.set_xlabel('Curvature κ [×10⁻³ mm⁻¹]', fontsize=11)
ax2.set_ylabel('Steel Force F_s [kN]', fontsize=11)
ax2.set_title('Steel Force vs Curvature', fontweight='bold')
ax2.grid(True, alpha=0.3)

ax3.plot(kappa_plot, N_plot, 'g-o', linewidth=2, markersize=6)
ax3.set_xlabel('Curvature κ [×10⁻³ mm⁻¹]', fontsize=11)
ax3.set_ylabel('Axial Force N [kN]', fontsize=11)
ax3.set_title('Axial Force vs Curvature', fontweight='bold')
ax3.grid(True, alpha=0.3)

ax4.plot(kappa_plot, M_plot, 'm-o', linewidth=2, markersize=6)
ax4.set_xlabel('Curvature κ [×10⁻³ mm⁻¹]', fontsize=11)
ax4.set_ylabel('Moment M [kNm]', fontsize=11)
ax4.set_title('Bending Moment vs Curvature', fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Parametric study complete:")
print(f"  Curvature range: {kappa_plot[0]:.3f} to {kappa_plot[-1]:.3f} × 10⁻³ mm⁻¹")
print(f"  Bottom strain (fixed): {eps_bottom_fixed:.6f}")
print(f"  Moment range: {min(M_plot):.2f} to {max(M_plot):.2f} kNm")

## 9. Validation: Pure Bending vs Bending with Axial Force

Compare two loading scenarios:
1. Pure bending (N ≈ 0)
2. Bending with axial compression

In [ ]:
# Scenario 1: Pure bending (adjust strains to achieve N ≈ 0)
kappa_1 = -0.000008
eps_bottom_1 = 0.0004  # Small tension

profile_1 = StressStrainProfile(cs, kappa_1, eps_bottom_1)
F_c_1, F_s_1, N_1, M_1 = profile_1.get_force_resultants()

print("Scenario 1: Pure Bending")
print(f"  Curvature: {kappa_1*1000:.3f} × 10⁻³ mm⁻¹")
print(f"  Bottom strain: {eps_bottom_1:.6f}")
print(f"  F_c: {F_c_1/1000:.2f} kN")
print(f"  F_s: {F_s_1/1000:.2f} kN")
print(f"  N: {N_1/1000:.2f} kN (should be ≈0)")
print(f"  M: {M_1/1e6:.2f} kNm")

In [ ]:
# Scenario 2: Bending with axial compression
kappa_2 = -0.000008
eps_bottom_2 = -0.0005  # Compression at bottom

profile_2 = StressStrainProfile(cs, kappa_2, eps_bottom_2)
F_c_2, F_s_2, N_2, M_2 = profile_2.get_force_resultants()

print("\nScenario 2: Bending with Axial Compression")
print(f"  Curvature: {kappa_2*1000:.3f} × 10⁻³ mm⁻¹")
print(f"  Bottom strain: {eps_bottom_2:.6f}")
print(f"  F_c: {F_c_2/1000:.2f} kN")
print(f"  F_s: {F_s_2/1000:.2f} kN")
print(f"  N: {N_2/1000:.2f} kN (compression)")
print(f"  M: {M_2/1e6:.2f} kNm")

In [ ]:
# Compare visualizations
fig = plt.figure(figsize=(14, 12))

# Scenario 1 plots
ax1_strain = plt.subplot(2, 2, 1)
ax1_stress = plt.subplot(2, 2, 2, sharey=ax1_strain)
profile_1.plot_stress_strain_profile(ax1_strain, ax1_stress, show_resultants=True)
ax1_strain.set_title(f'Scenario 1: Pure Bending (N={N_1/1000:.1f} kN)', fontsize=12, fontweight='bold')

# Scenario 2 plots
ax2_strain = plt.subplot(2, 2, 3)
ax2_stress = plt.subplot(2, 2, 4, sharey=ax2_strain)
profile_2.plot_stress_strain_profile(ax2_strain, ax2_stress, show_resultants=True)
ax2_strain.set_title(f'Scenario 2: Bending + Compression (N={N_2/1000:.1f} kN)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Summary and Validation

Key findings from this notebook:

1. **Plane Section Assumption**: The strain distribution is linear across the section height
2. **Material Nonlinearity**: Stress distribution is nonlinear due to material constitutive laws
3. **Force Equilibrium**: F_c + F_s = N (verified numerically)
4. **Moment Calculation**: Accurate integration of stress distributions
5. **Visualization**: Coupled strain-stress plots provide complete understanding

The `StressStrainProfile` class successfully implements:
- ✓ Strain profile calculation
- ✓ Stress profile calculation
- ✓ Force resultant computation
- ✓ Comprehensive visualization
- ✓ Integration with Phase 1 (matmod) and Phase 2 (cs_design) modules